# 06 — JSONOutputParser

Neste notebook vamos aprender a utilizar o **`JsonOutputParser`** do LangChain para transformar a resposta da LLM em uma estrutura JSON que podemos manipular no Python.

Até agora, normalmente recebíamos texto:

```text
LLM → texto
```

Agora queremos:

```text
LLM → JSON → Python
```

Isso é importante porque aplicações reais geralmente precisam trabalhar com os dados gerados pela IA, e não apenas exibir texto na tela.

## 1. Por que usar saída JSON?

Imagine que perguntamos à IA sobre um conceito:

```text
Explique RAG.
```

Uma resposta textual pode variar bastante.

Para uma aplicação, seria muito mais útil receber algo como:

```json
{
  "conceito": "RAG",
  "definicao": "...",
  "nivel": "intermediário",
  "exemplo": "..."
}
```

Assim o nosso código consegue acessar cada informação individualmente.

## 2. Importando os componentes

Vamos utilizar três componentes:

- `ChatOllama`: comunicação com o Qwen.
- `PromptTemplate`: criação do prompt.
- `JsonOutputParser`: transformação da resposta JSON em uma estrutura Python.

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser

## 3. Configurando o Qwen

Vamos continuar utilizando o modelo local `qwen3:4b` através do Ollama.

Neste exemplo também usamos `format="json"` para solicitar ao Ollama que produza a resposta em formato JSON.

In [ ]:
llm = ChatOllama(
    model="qwen3:4b",
    base_url="http://localhost:11434",
    format="json"
)

### Por que `format="json"`?

O Ollama possui suporte para solicitar que o modelo gere JSON.

Isso ajuda a garantir que a resposta seja compatível com o formato que queremos processar.

Mesmo assim, ainda é importante instruir o modelo no prompt sobre quais campos esperamos.

## 4. Criando o JsonOutputParser

Agora criamos o parser:

In [ ]:
parser = JsonOutputParser()

O `JsonOutputParser` é responsável por interpretar uma resposta JSON da LLM e transformá-la em uma estrutura Python.

Na prática, podemos trabalhar com o resultado como um `dict`.

## 5. Criando o prompt

Agora vamos pedir explicitamente que a IA retorne algumas propriedades específicas.

In [ ]:
prompt = PromptTemplate.from_template(
    """
    Analise o conceito abaixo e responda em formato JSON.

    Conceito: {conceito}

    O JSON deve conter exatamente estas propriedades:
    - conceito: nome do conceito
    - definicao: uma definição curta
    - nivel: nível de dificuldade (iniciante, intermediário ou avançado)
    - exemplo: um exemplo prático

    Retorne somente o JSON, sem markdown.
    """
)

### Atenção ao formato

Estamos ensinando ao modelo qual estrutura queremos.

Esperamos algo semelhante a:

```json
{
  "conceito": "RAG",
  "definicao": "...",
  "nivel": "intermediário",
  "exemplo": "..."
}
```

A estrutura exata do conteúdo será gerada pelo modelo.

## 6. Criando a chain com LCEL

Agora podemos conectar os três componentes usando o operador `|`:

```text
PromptTemplate
      ↓
ChatOllama
      ↓
JsonOutputParser
```

O resultado do modelo passa diretamente para o parser.

In [ ]:
chain = prompt | llm | parser

## 7. Executando a chain

Vamos pedir para a IA analisar o conceito de `RAG`.

In [ ]:
resultado = chain.invoke({
    "conceito": "RAG"
})

## 8. Visualizando o resultado

In [ ]:
print(resultado)

Agora podemos verificar o tipo retornado pelo parser.

In [ ]:
print(type(resultado))

Normalmente veremos:

```text
<class 'dict'>
```

Ou seja, o JSON produzido pela IA foi convertido para um dicionário Python.

## 9. Acessando campos individualmente

Como o resultado é um dicionário, podemos acessar cada propriedade:

In [ ]:
print("Conceito:", resultado["conceito"])
print("Definição:", resultado["definicao"])
print("Nível:", resultado["nivel"])
print("Exemplo:", resultado["exemplo"])

Isso demonstra uma das principais vantagens da saída estruturada.

Em vez de tratar toda a resposta como uma única string, nossa aplicação consegue trabalhar com cada informação separadamente.

## 10. Testando outros conceitos

Como nosso prompt possui uma variável, podemos reutilizar a mesma chain para diferentes conceitos.

In [ ]:
conceitos = ["LangChain", "embeddings", "agents", "machine learning"]

for conceito in conceitos:
    resultado = chain.invoke({
        "conceito": conceito
    })

    print(f"\n--- {conceito} ---")
    print("Definição:", resultado["definicao"])
    print("Nível:", resultado["nivel"])

## 11. Fluxo completo

Neste exercício construímos o seguinte fluxo:

```text
conceito
   ↓
PromptTemplate
   ↓
Qwen / Ollama
   ↓
JSON
   ↓
JsonOutputParser
   ↓
dict Python
```

Esse padrão aparece bastante em aplicações que precisam consumir respostas de LLMs programaticamente.

## 12. Código completo

Agora juntamos tudo:

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser

llm = ChatOllama(
    model="qwen3:4b",
    base_url="http://localhost:11434",
    format="json"
)

parser = JsonOutputParser()

prompt = PromptTemplate.from_template(
    """
    Analise o conceito abaixo e responda em formato JSON.

    Conceito: {conceito}

    O JSON deve conter exatamente estas propriedades:
    - conceito: nome do conceito
    - definicao: uma definição curta
    - nivel: nível de dificuldade (iniciante, intermediário ou avançado)
    - exemplo: um exemplo prático

    Retorne somente o JSON, sem markdown.
    """
)

chain = prompt | llm | parser

resultado = chain.invoke({
    "conceito": "RAG"
})

print(resultado)
print(resultado["definicao"])